In [ ]:
# P1-fix: os caminhos deste notebook são RELATIVOS a EXIST_VIDEOS/ (plano 11 §3).
import os
from pathlib import Path

_TARGET = Path("/home/a67606/Thesis/EXIST_VIDEOS")
if Path.cwd() != _TARGET:
    assert _TARGET.is_dir(), f"diretório do projeto não encontrado: {_TARGET}"
    os.chdir(_TARGET)
assert Path("Dataset").is_dir(), f"cwd errado: {Path.cwd()} (sem Dataset/)"
assert Path("video_core.py").is_file(), "video_core.py não está no cwd"
assert Path("video_core_clip.py").is_file(), "video_core_clip.py não está no cwd"
print("cwd:", Path.cwd())

import sys
_ORIG_STDOUT = globals().get("_ORIG_STDOUT", sys.stdout)

class _Tee:
    def __init__(self, path, term):
        self.f = open(path, "w", buffering=1, encoding="utf-8")
        self.term = term
    def write(self, s):
        self.term.write(s); self.f.write(s); return len(s)
    def flush(self):
        self.term.flush(); self.f.flush()
    def isatty(self):
        return False

Path("analysis").mkdir(exist_ok=True)
LOG_PATH = "analysis/probe_head_training_report.txt"
sys.stdout = _Tee(LOG_PATH, _ORIG_STDOUT)
print("[log] stdout duplicado para", LOG_PATH,
      "\n[log] stderr NAO e capturado — se houver traceback, copiar da tela")


# Probing das 3 heads sobre trunk congelado — CLIP e B15

**Pré-registro:** `improvements/21_HEAD_PROBING_PREREGISTRATION.md` — ler antes de mexer
em qualquer parâmetro deste notebook. Decisões travadas lá (critério de parada, LR
escolhido por convergência de treino e não por métrica, leitura A/B=controle vs
C=discriminador) não são reabertas aqui.

**O que este notebook faz:** para CADA trunk (CLIP-final e B15-oficial), extrai o vetor
de fusão do TREINO (2.019) e reusa/extrai o do VAL (505), depois treina 3 probes lineares
independentes (`head_a`, `head_b`, `head_c`) sobre features CONGELADAS, cada uma parando
pela própria loss no val inteiro. Reporta original vs probe, por trunk, por head, nas 6
métricas ICM-Norm.

**O que este notebook NÃO faz:** não muda o trunk. Não decide sozinho se a variação é
adotada — isso é decisão do autor, doc 21 §6.

## 0. Configuração — os dois trunks

In [ ]:
import json, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import namedtuple
from transformers import XLMRobertaTokenizer, TrainingArguments

import video_core as vc
import video_core_clip as vcc

EvalPred = namedtuple("EvalPred", ["predictions", "label_ids"])

# ⚠ AJUSTAR os dois RUN_DIR abaixo se os nomes reais divergirem — os dois já foram
# confirmados existindo no mount (2026-08-15): diretórios com meta.json +
# trainable.safetensors + final_metrics.json (NÃO um .safetensors solto de
# epoch_states/ — vcc.load_run_clip/vc.load_run esperam o diretório da run inteira).
TRUNKS = {
    "CLIP": {
        "run_dir": Path("run_B2_bio_viewer_seed42_CLIP"),
        "kind": "clip",
    },
    "B15": {
        "run_dir": Path("run_B2_bio_viewer_seed42_B15"),
        "kind": "videomae",
    },
}

HEAD_OUT_DIM = {"a": vc.NUM_TASK_A, "b": vc.NUM_TASK_B, "c": vc.NUM_TASK_C}
HEAD_LOSS_FN = {"a": vc.loss_task_a, "b": vc.loss_task_b, "c": vc.loss_task_c}
FUSION_DIM = vc.FUSION_HIDDEN_DIM

print("HEAD_OUT_DIM:", HEAD_OUT_DIM, "| FUSION_HIDDEN_DIM:", FUSION_DIM)

full_df = pd.read_json("Dataset/training/EXIST2026_training.json").transpose()
train_df, val_df = vc.build_frozen_split(full_df)
tokenizer = XLMRobertaTokenizer.from_pretrained(vc.XLMR_NAME)
print(f"treino={len(train_df)} val={len(val_df)} (split congelado seed {vc.SPLIT_SEED})")


## 1. Carregar cada trunk e montar seu dataset

`kind="clip"` usa `vcc.load_run_clip` + `vcc.EXISTVideoDatasetCLIP` (renormalização afim
para estatística do CLIP). `kind="videomae"` usa `vc.load_run` + `vc.EXISTVideoDataset`
(padrão original). Cada trunk fica com seu próprio `(model, train_ds, val_ds)` — nada é
compartilhado entre eles.

In [ ]:
def load_trunk(name, spec):
    run_dir = spec["run_dir"]
    if spec["kind"] == "clip":
        model, scaler = vcc.load_run_clip(run_dir, init_seed=42)
        ds_cls = vcc.EXISTVideoDatasetCLIP
    elif spec["kind"] == "videomae":
        model, scaler = vc.load_run(run_dir, init_seed=42)
        ds_cls = vc.EXISTVideoDataset
    else:
        raise ValueError(spec["kind"])

    with open(run_dir / "meta.json") as f:
        meta = json.load(f)
    with open(run_dir / "final_metrics.json") as f:
        final = json.load(f)

    train_ds = ds_cls(train_df, tokenizer, scaler, vc.frames_dir("train"),
                      vc.audio_dir("train"), max_length=vc.MAX_LENGTH)
    val_ds = ds_cls(val_df, tokenizer, scaler, vc.frames_dir("train"),
                    vc.audio_dir("train"), max_length=vc.MAX_LENGTH)

    model.eval()
    print(f"[{name}] carregado de {run_dir} | kind={spec['kind']} | "
          f"epoch_selection={meta.get('epoch_selection', meta.get('selected_epoch'))}")
    return {"model": model, "train_ds": train_ds, "val_ds": val_ds,
            "meta": meta, "final": final}

trunks = {name: load_trunk(name, spec) for name, spec in TRUNKS.items()}


## 2. Extração de features — TREINO (nova) e VAL (reusa cache se existir)

`eval()` explícito, hook na **saída** de `model.fusion` inteiro (não na entrada de
`fusion[0]`) — captura o vetor **pós** `Linear→GELU→Dropout`, 1024-d
(`FUSION_HIDDEN_DIM`), que é exatamente o que `head_a/b/c` consomem
(`video_core.py`: `logits_a = self.head_a(fused)`, etc.). Diferente dos probes causais
anteriores (`probe_video_contribution_clip.ipynb`), que capturam a **entrada** de
`fusion[0]` (2944-d, pré-fusão) porque ali o objetivo é perturbar por slot de
modalidade — aqui o objetivo é treinar heads sobre o que elas realmente recebem.

**Caveat travado no doc 21 §1.1: extração SEM dropout** (mesmo que a head original tenha
visto features com dropout ativo durante o treino end-to-end).

In [ ]:
def extract_fusion(model, dataset, tag):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    assert device == "cuda", "extração exige GPU — rodar na VM"
    model.to(device).eval()

    args = TrainingArguments(output_dir="/tmp/probe_head_extract",
                             per_device_eval_batch_size=2, bf16=True,
                             remove_unused_columns=False, report_to="none",
                             label_names=["labels_a", "labels_b", "labels_c"])
    trainer = vc.MultiTaskTrainer(model=model, args=args,
                                  compute_metrics=vc.compute_metrics,
                                  data_collator=vc.video_collate)
    trainer.label_names = ["labels_a", "labels_b", "labels_c"]

    buf = []
    def hook(_m, _inp, out):
        # saída de model.fusion INTEIRO (Linear->GELU->Dropout), 1024-d — o que as
        # heads consomem. NÃO a entrada de fusion[0] (2944-d, pré-fusão).
        buf.append(out.detach().float().cpu())
    h = model.fusion.register_forward_hook(hook)
    try:
        out = trainer.predict(dataset)
    finally:
        h.remove()

    logits_a, logits_b, logits_c = vc._unpack_predictions(out.predictions)
    lab_a, lab_b, lab_c = out.label_ids
    result = {
        "fused": torch.cat(buf, dim=0),
        "logits_a": np.asarray(logits_a), "logits_b": np.asarray(logits_b),
        "logits_c": np.asarray(logits_c),
        "labels_a": np.asarray(lab_a), "labels_b": np.asarray(lab_b),
        "labels_c": np.asarray(lab_c),
    }
    print(f"[{tag}] fused {tuple(result['fused'].shape)}")
    assert result["fused"].shape[1] == FUSION_DIM, (
        f"esperava {FUSION_DIM}-d (FUSION_HIDDEN_DIM), veio {result['fused'].shape[1]} "
        f"— o hook capturou o lugar errado do módulo fusion")
    return result


CACHE_ROOT = Path("analysis")
for name, t in trunks.items():
    run_name = TRUNKS[name]["run_dir"].name

    train_cache = CACHE_ROOT / f"probe_train_fused_{run_name}.pt"
    if train_cache.exists():
        t["train_cache"] = torch.load(train_cache, weights_only=False)
        print(f"[{name}][train] cache carregado de {train_cache}")
    else:
        t["train_cache"] = extract_fusion(t["model"], t["train_ds"], f"{name}/train")
        torch.save(t["train_cache"], train_cache)
        print(f"[{name}][train] cache gravado em {train_cache}")

    # ⚠ Cache do VAL para este notebook é DIFERENTE do cache dos probes causais
    # (aquele guarda o vetor pré-fusão 2944-d; este guarda o pós-fusão 1024-d) — nome de
    # arquivo distinto para nunca colidir nem ser reusado por engano entre os dois.
    val_cache_path = CACHE_ROOT / f"probe_head_val_fused_{run_name}.pt"
    if val_cache_path.exists():
        t["val_cache"] = torch.load(val_cache_path, weights_only=False)
        print(f"[{name}][val] cache carregado de {val_cache_path}")
    else:
        t["val_cache"] = extract_fusion(t["model"], t["val_ds"], f"{name}/val")
        torch.save(t["val_cache"], val_cache_path)
        print(f"[{name}][val] cache gravado em {val_cache_path}")


## 3. Smoke test de LR — escolhido por convergência de TREINO, não por métrica

Duas ou três taxas candidatas, critério: qual reduz a loss de treino mais rápido/estável
nas primeiras iterações. **Nunca escolher por ICM ou por loss de val** — isso
contaminaria a escolha de hiperparâmetro com a métrica reportada.

In [ ]:
def train_probe(fused_train, labels_train, fused_val, labels_val,
                out_dim, loss_fn, lr, max_epochs, init_seed, device):
    vc.set_all_seeds(init_seed)
    head = nn.Linear(FUSION_DIM, out_dim).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=0.01)

    x_tr = fused_train.to(device)
    y_tr = torch.from_numpy(labels_train).to(device)
    x_va = fused_val.to(device)
    y_va = torch.from_numpy(labels_val).to(device)

    best_state, best_val_loss, best_epoch = None, float("inf"), -1
    train_losses, val_losses = [], []
    for epoch in range(max_epochs):
        head.train()
        opt.zero_grad()
        logits_tr = head(x_tr)
        if out_dim == 1:
            logits_tr = logits_tr.squeeze(-1)
        loss_tr = loss_fn(logits_tr, y_tr)
        loss_tr.backward()
        opt.step()

        head.eval()
        with torch.no_grad():
            logits_va = head(x_va)
            if out_dim == 1:
                logits_va = logits_va.squeeze(-1)
            loss_va = loss_fn(logits_va, y_va).item()

        train_losses.append(loss_tr.item())
        val_losses.append(loss_va)
        if loss_va < best_val_loss:
            best_val_loss = loss_va
            best_epoch = epoch
            best_state = copy.deepcopy(head.state_dict())

    return {"best_state": best_state, "best_val_loss": best_val_loss,
            "best_epoch": best_epoch, "train_losses": train_losses,
            "val_losses": val_losses}


device = "cuda" if torch.cuda.is_available() else "cpu"
MAX_EPOCHS = 300

# smoke test: LR candidato pela convergência do TREINO (head C do trunk CLIP, descartável)
_t = trunks["CLIP"]
_ft = _t["train_cache"]["fused"]
_lc = _t["train_cache"]["labels_c"]
print(f"{'lr':>8s}{'loss@10':>12s}{'loss@50':>12s}{'loss@150':>12s}")
for lr in (1e-2, 1e-3, 1e-4):
    r = train_probe(_ft, _lc, _t["val_cache"]["fused"], _t["val_cache"]["labels_c"],
                    HEAD_OUT_DIM["c"], HEAD_LOSS_FN["c"], lr, 150, vc.INIT_SEED, device)
    tl = r["train_losses"]
    print(f"{lr:8.0e}{tl[9]:12.4f}{tl[49]:12.4f}{tl[149]:12.4f}")

PROBE_LR = 1e-3   # <<< ajustar após ver a tabela acima, se necessário — pela CONVERGÊNCIA
print(f"\nPROBE_LR escolhido = {PROBE_LR}")


## 4. Treinar as 3 probes em cada trunk

`vc.set_all_seeds(INIT_SEED)` re-chamado imediatamente antes de cada `nn.Linear` novo
(dentro de `train_probe`) — ordem A→B→C, sempre re-seed antes de cada uma.

In [ ]:
results = {}
for name, t in trunks.items():
    results[name] = {}
    for head in ("a", "b", "c"):
        r = train_probe(
            t["train_cache"]["fused"], t["train_cache"][f"labels_{head}"],
            t["val_cache"]["fused"], t["val_cache"][f"labels_{head}"],
            HEAD_OUT_DIM[head], HEAD_LOSS_FN[head], PROBE_LR, MAX_EPOCHS,
            vc.INIT_SEED, device)
        results[name][head] = r
        print(f"[{name}][head_{head}] best_epoch={r['best_epoch']:3d}  "
              f"best_val_loss={r['best_val_loss']:.4f}")


## 5. Comparação original vs probe — métricas ICM-Norm

Para cada head, os logits das OUTRAS DUAS heads ficam no valor ORIGINAL (do
`final_metrics.json`/forward real) — só a head sob teste é trocada pela probe. Isso
isola o efeito de cada head individualmente, sem misturar mudanças simultâneas.

In [ ]:
METRIC_KEYS = [
    "task_a_icmnorm_hard", "task_a_icmsoftnorm_soft",
    "task_b_icmnorm_hard", "task_b_icmsoftnorm_soft",
    "task_c_icmnorm_hard", "task_c_icmsoftnorm_soft",
]

def metrics_with_probe(t, probe_head):
    vc_ = t["val_cache"]
    logits = {"a": torch.from_numpy(vc_["logits_a"]).float(),
             "b": torch.from_numpy(vc_["logits_b"]).float(),
             "c": torch.from_numpy(vc_["logits_c"]).float()}
    head_mod = nn.Linear(FUSION_DIM, HEAD_OUT_DIM[probe_head])
    head_mod.load_state_dict(results[t["_name"]][probe_head]["best_state"])
    head_mod.eval()
    with torch.no_grad():
        out = head_mod(vc_["fused"])
        if HEAD_OUT_DIM[probe_head] == 1:
            out = out.squeeze(-1)
    logits[probe_head] = out
    ep = EvalPred((logits["a"].numpy(), logits["b"].numpy(), logits["c"].numpy()),
                  (vc_["labels_a"], vc_["labels_b"], vc_["labels_c"]))
    m = vc.compute_metrics(ep)
    return {k: float(m[k]) for k in METRIC_KEYS}


rows = []
for name, t in trunks.items():
    t["_name"] = name
    orig = {k: float(t["final"][f"eval_{k}"]) for k in METRIC_KEYS}
    rows.append({"trunk": name, "variante": "original", **orig})
    for head in ("a", "b", "c"):
        m = metrics_with_probe(t, head)
        rows.append({"trunk": name, "variante": f"probe_{head}", **m})

table = pd.DataFrame(rows)
pd.set_option("display.width", 160)
print(table.to_string(index=False))


## 6. Leitura pré-registada (doc 21 §1.4) — aplicada, não reformulada

In [ ]:
print("=" * 90)
print("LEITURA PRE-REGISTADA")
print("=" * 90)
for name, t in trunks.items():
    orig_row = table[(table.trunk == name) & (table.variante == "original")].iloc[0]
    print(f"\ntrunk = {name}")

    controls_ok = True
    for head, metric_hard, metric_soft in (
        ("a", "task_a_icmnorm_hard", "task_a_icmsoftnorm_soft"),
        ("b", "task_b_icmnorm_hard", "task_b_icmsoftnorm_soft"),
    ):
        probe_row = table[(table.trunk == name) & (table.variante == f"probe_{head}")].iloc[0]
        d_hard = probe_row[metric_hard] - orig_row[metric_hard]
        d_soft = probe_row[metric_soft] - orig_row[metric_soft]
        print(f"  head_{head} [CONTROLE (A/B)]  D_hard={d_hard:+.4f}  D_soft={d_soft:+.4f}")
        if abs(d_hard) > 0.03 or abs(d_soft) > 0.03:
            controls_ok = False
            print(f"    ⚠ controle moveu mais que 0.03 — desconfiar do protocolo "
                  f"(bug de extração/split), não comemorar.")

    probe_row = table[(table.trunk == name) & (table.variante == "probe_c")].iloc[0]
    d_hard = probe_row["task_c_icmnorm_hard"] - orig_row["task_c_icmnorm_hard"]
    d_soft = probe_row["task_c_icmsoftnorm_soft"] - orig_row["task_c_icmsoftnorm_soft"]
    print(f"  head_c [DISCRIMINADOR (C)]  D_hard={d_hard:+.4f}  D_soft={d_soft:+.4f}")

    # GATE: o veredito de C só é válido se os controles A/B tiverem empatado (dentro da
    # tolerância). Se A/B falharam, o protocolo inteiro não é confiável — imprimir
    # INCONCLUSIVO, nunca deixar o veredito substantivo (sub-treinada / sem sinal)
    # sobreviver sem essa checagem. Ver improvements/21_HEAD_PROBING_PREREGISTRATION.md,
    # seção "Resultado".
    if not controls_ok:
        print("    -> INCONCLUSIVO — controles A/B falharam, protocolo de probing não é "
              "confiável para decidir sobre C nesta rodada. NÃO ler como 'C sem sinal' "
              "nem como 'C sub-treinada'. Ver doc 21, seção Resultado, para o mecanismo "
              "provável (probe parando cedo demais em BCE antes de calibrar p/ ICM).")
    elif d_hard > 0.03 or d_soft > 0.03:
        print("    -> C_probe >> C_original: head sub-treinada, ganho barato, "
              "caso fechado nessa direção.")
    else:
        print("    -> C_probe ~= C_original: representação congelada não "
              "carrega mais sinal de C. Só resta repesar LOSS_W_C no treino "
              "(opção 5 do doc 20), retreino completo do trunk.")
